# RSNA Knee — V7B CoAtNet Raptor single method (safe T4×2)

V7B reproduces the high-scoring public CoAtNet principle as one standalone model, with a safer memory profile for Kaggle T4×2:

- Fixed 44-slice corpus at 336 px: five slots `(12,10,8,6,8)`, 140 mm crop, 15–85% coverage.
- Random **6-window** adjacent 2.5D bag on every training epoch; deterministic **12-window** validation bag.
- `coatnet_rmlp_2_rw_384.sw_in12k_ft_in1k`, resized to 384 and fully fine-tuned end-to-end.
- Separate attention distribution and classifier vector for each of 12 findings.
- Pilkwang `report_labels_v2` × Steven `llm_labels_v4_blend` 1:1 soft supervision.
- The 58 gold studies are validation-only and never enter the optimizer.
- Top-three epoch weight averaging, but inference loads exactly one selected state: no architecture ensemble.
- Kaggle T4×2 uses **1 study/GPU** (global batch 2), **validation batch 1**, gradient checkpointing, and explicit CUDA cleanup before validation.
- LR scheduler advances only when AMP actually performs an optimizer step, preventing the `scheduler.step()`-before-`optimizer.step()` warning after overflow.

Attach the competition, both Raptor corpus parts, and both pseudo-label datasets. Select **GPU T4 x2**.


In [ ]:
from __future__ import annotations

import gc, hashlib, json, math, os, random, re, sys, time, traceback, unicodedata
from dataclasses import asdict, dataclass
from pathlib import Path
from typing import Dict, List, Optional, Tuple

import numpy as np
import pandas as pd
import pydicom
import torch
import torch.nn as nn
import torch.nn.functional as F
from pydicom.dataset import FileDataset, FileMetaDataset
from pydicom.uid import ExplicitVRLittleEndian, generate_uid
from sklearn.metrics import roc_auc_score
from torch.utils.data import DataLoader, Dataset

try:
    import timm
except Exception:
    timm = None

TARGETS = [
    "ACL", "MCL", "Medial Meniscus", "Lateral Meniscus",
    "Medial OA", "Lateral OA", "PF OA", "Effusion",
    "Synovitis", "Baker's", "Contusion", "Fracture",
]
UID = "StudyInstanceUID"


@dataclass
class CFG:
    seed: int = 42
    smoke: bool = os.getenv("RSNA_SMOKE_TEST", "0") == "1"
    comp_root: Optional[str] = os.getenv("RSNA_COMP_ROOT")
    output_dir: str = os.getenv("RSNA_TRAIN_OUTPUT", "/kaggle/working/rsna_knee_v7b_coatnet")
    pilkwang_label_path: Optional[str] = os.getenv("RSNA_PILKWANG_LABELS")
    steven_label_path: Optional[str] = os.getenv("RSNA_STEVEN_LABELS")
    steven_blend_alpha: float = 0.50
    require_llm_labels: bool = True
    pseudo_temperature: float = 1.0
    unknown_weight: float = 0.03

    # Exact public Raptor stack contract.
    tokens_per_slot: Tuple[int, ...] = (12, 10, 8, 6, 8)
    train_tokens_per_slot: Tuple[int, ...] = (12, 10, 8, 6, 8)
    corpus_size: int = 336
    crop_mm: float = 140.0
    coverage: Tuple[float, float] = (0.15, 0.85)

    arch: str = "coatnet_rmlp_2_rw_384.sw_in12k_ft_in1k"
    resolution: int = 384
    train_windows: int = 6
    eval_windows: int = 12
    epochs: int = 16
    batch_per_gpu: int = 1
    valid_batch_size: int = 1
    max_gpus: int = 2
    workers: int = 2
    backbone_lr: float = 3e-5
    head_lr: float = 1e-3
    weight_decay: float = 0.02
    dropout: float = 0.20
    top_k: int = 3
    grad_checkpointing: bool = True
    intensity_jitter: float = 0.20


cfg = CFG()
if cfg.smoke:
    cfg.output_dir = os.getenv("RSNA_TRAIN_OUTPUT", str(Path.cwd() / "smoke_v7b_train"))
    cfg.corpus_size = 64
    cfg.resolution = 64
    cfg.train_windows = 4
    cfg.eval_windows = 6
    cfg.epochs = 2
    cfg.batch_per_gpu = 4
    cfg.valid_batch_size = 1
    cfg.workers = 0
    cfg.top_k = 2
    cfg.require_llm_labels = False

OUT = Path(cfg.output_dir)
OUT.mkdir(parents=True, exist_ok=True)
DEVICE = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
N_GPU = min(cfg.max_gpus, torch.cuda.device_count()) if torch.cuda.is_available() else 0
EFFECTIVE_GPU = max(1, N_GPU)
BATCH_SIZE = cfg.batch_per_gpu * EFFECTIVE_GPU
torch.set_float32_matmul_precision("high")
if torch.cuda.is_available():
    torch.backends.cudnn.benchmark = True

random.seed(cfg.seed); np.random.seed(cfg.seed); torch.manual_seed(cfg.seed)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(cfg.seed)
print(json.dumps(asdict(cfg), indent=2, ensure_ascii=False))
print("device:", DEVICE, "physical GPUs:", torch.cuda.device_count(),
      "training GPUs:", N_GPU, "global batch:", BATCH_SIZE)
if torch.cuda.is_available():
    print("GPU names:", [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])
if not cfg.smoke:
    assert sum(cfg.tokens_per_slot) == 44
    assert cfg.arch == "coatnet_rmlp_2_rw_384.sw_in12k_ft_in1k"
    if timm is None:
        raise ImportError("timm is required; Kaggle's standard GPU image includes it")

## 1. Competition data and smoke fixture

In [ ]:
SLOT_SPECS = [
    # name, plane, desired recovered contrast (header recovery has CSV fallback)
    ("sag_fluid", "Sagittal", "fluid"),
    ("sag_struct", "Sagittal", "struct"),
    ("cor_fluid", "Coronal", "fluid"),
    ("cor_struct", "Coronal", "struct"),
    ("ax_fluid", "Axial", "fluid"),
]

assert len(SLOT_SPECS) == len(cfg.tokens_per_slot) == len(cfg.train_tokens_per_slot)
assert all(0 < train_n <= dense_n for train_n, dense_n in zip(
    cfg.train_tokens_per_slot, cfg.tokens_per_slot
))
if not cfg.smoke:
    assert sum(cfg.tokens_per_slot) == 44


def _write_synthetic_dicom(path: Path, value: int, z: float, plane: str, laterality: str) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    meta = FileMetaDataset()
    meta.MediaStorageSOPClassUID = generate_uid()
    meta.MediaStorageSOPInstanceUID = generate_uid()
    meta.TransferSyntaxUID = ExplicitVRLittleEndian
    ds = FileDataset(str(path), {}, file_meta=meta, preamble=b"\0" * 128)
    ds.SOPClassUID = meta.MediaStorageSOPClassUID
    ds.SOPInstanceUID = meta.MediaStorageSOPInstanceUID
    ds.Rows = ds.Columns = 64
    ds.SamplesPerPixel = 1
    ds.PhotometricInterpretation = "MONOCHROME2"
    ds.BitsAllocated = ds.BitsStored = 16
    ds.HighBit = 15
    ds.PixelRepresentation = 0
    ds.PixelSpacing = [0.6, 0.6]
    ds.InstanceNumber = int(z + 20)
    ds.Laterality = laterality
    if plane == "Sagittal":
        ds.ImageOrientationPatient = [0, 1, 0, 0, 0, -1]
        ds.ImagePositionPatient = [z, 0, 0]
    elif plane == "Coronal":
        ds.ImageOrientationPatient = [1, 0, 0, 0, 0, -1]
        ds.ImagePositionPatient = [0, z, 0]
    else:
        ds.ImageOrientationPatient = [1, 0, 0, 0, 1, 0]
        ds.ImagePositionPatient = [0, 0, z]
    yy, xx = np.mgrid[:64, :64]
    arr = value + 300 * np.exp(-((xx - 32) ** 2 + (yy - 32) ** 2) / 250)
    arr += 40 * np.sin((xx + z) / 6)
    ds.PixelData = np.clip(arr, 0, 65535).astype(np.uint16).tobytes()
    ds.save_as(path, write_like_original=False)


def make_smoke_competition(root: Path, n_train: int = 16, n_test: int = 4) -> Path:
    root.mkdir(parents=True, exist_ok=True)
    train_rows, test_rows, train_series, test_series = [], [], [], []
    reports = [
        "Complete ACL tear with moderate joint effusion. No fracture.",
        "Medial meniscus tear. Mild medial compartment osteoarthritis.",
        "No ligament or meniscal injury. Small Baker cyst.",
        "Bone marrow contusion and synovitis. No fracture.",
    ]
    for split, count in (("train", n_train), ("test", n_test)):
        for i in range(count):
            study = f"{split}.study.{i:03d}"
            if split == "train":
                row = {UID: study, "Report": reports[i % len(reports)] + f" template {i // 4}"}
                vals = np.array([(i + j) % 3 == 0 for j in range(len(TARGETS))], np.float32)
                for j, target in enumerate(TARGETS):
                    row[target] = float(vals[j]) if i < 8 else np.nan
                train_rows.append(row)
            else:
                test_rows.append({UID: study})
            for s, (_, plane, desired) in enumerate(SLOT_SPECS):
                fluid = fs = int(desired == "fluid")
                series = f"{study}.series.{s}"
                rec = {UID: study, "SeriesInstanceUID": series, "Fluid_Sensitive": fluid,
                       "Fat_Suppression": fs, "Anatomical_Plane": plane}
                (train_series if split == "train" else test_series).append(rec)
                folder = root / f"{split}_series" / study / series
                for k in range(7):
                    _write_synthetic_dicom(folder / f"{k:03d}.dcm", 100 + 7 * i + 11 * s,
                                           float(k), plane, "R" if i % 2 else "L")
    pd.DataFrame(train_rows).to_csv(root / "train.csv", index=False)
    pd.DataFrame(test_rows).to_csv(root / "test.csv", index=False)
    pd.DataFrame(train_series).to_csv(root / "train_series.csv", index=False)
    pd.DataFrame(test_series).to_csv(root / "test_series.csv", index=False)
    sample = pd.DataFrame(test_rows)
    for t in TARGETS:
        sample[t] = 0.5
    sample.to_csv(root / "sample_submission.csv", index=False)
    return root


def find_comp_root() -> Path:
    if cfg.comp_root:
        p = Path(cfg.comp_root)
        if (p / "train.csv").is_file():
            return p
    if cfg.smoke:
        p = Path.cwd() / "smoke_rsna_data"
        return make_smoke_competition(p)
    candidates = [
        Path("/kaggle/input/competitions/rsna-knee-abnormality-detection"),
        Path("/kaggle/input/rsna-knee-abnormality-detection"),
        Path.cwd() / "data",
        Path.cwd(),
    ]
    for p in candidates:
        if (p / "train.csv").is_file() and (p / "train_series.csv").is_file():
            return p
    base = Path("/kaggle/input")
    if base.is_dir():
        for p in base.iterdir():
            if p.is_dir() and (p / "train.csv").is_file() and (p / "train_series").is_dir():
                return p
    raise FileNotFoundError("Attach the RSNA Knee competition data or set RSNA_COMP_ROOT")


ROOT = find_comp_root()
train_df = pd.read_csv(ROOT / "train.csv", dtype={UID: str})
series_df = pd.read_csv(ROOT / "train_series.csv", dtype={UID: str, "SeriesInstanceUID": str})
for t in TARGETS:
    train_df[t] = pd.to_numeric(train_df[t], errors="coerce")
print("root:", ROOT)
print("train studies / series:", len(train_df), len(series_df))

## 2. Your Pilkwang × Steven-v4 pseudo labels

In [ ]:
_PRE = str.maketrans({"ı": "i", "İ": "i", "ß": "ss", "đ": "d", "ø": "o"})


def norm_report(x: object) -> str:
    if not isinstance(x, str):
        return ""
    x = unicodedata.normalize("NFKD", x.translate(_PRE).lower())
    x = "".join(c for c in x if not unicodedata.combining(c))
    x = re.sub(r"[_/\\-]+", " ", x)
    return re.sub(r"\s+", " ", x).strip()


NEG = re.compile(
    r"\b(no|not|without|absen\w*|negative for|sin|no hay|aucun\w*|sans|geen|zonder|"
    r"keine?\w*|ohne|yok\w*|saptanmadi|nema|bez|δεν|χωρις|без|няма)\b"
)
UNC = re.compile(r"\b(possible|probable|suspicious|suspected|may|equivocal|muhtemel|olasi|πιθαν|възможно)\b")
NORMAL = re.compile(r"\b(intact|normal|preserved|unremarkable|korunmus|regelrecht|φυσιολογικ|нормал)\w*")
PATH = re.compile(
    r"tear|torn|ruptur|injur|sprain|lesion|degenerat|discontin|rotura|desgarro|dechir|"
    r"scheur|riss|yirtik|puknu|ρηξ|разкъс|edema|oedema|signal alteration|high signal"
)
OA = re.compile(
    r"osteoarth|arthros|gonarth|chondropath|chondromal|condropat|cartilage (loss|thinning)|"
    r"osteophyt|joint space narrowing|artroz|arthrose|χονδρ|артроз|хондропат"
)
LOW = re.compile(r"\b(trace|small|minimal|mild|slight|tiny|leve|hafif|blago|gering|ηπι|лек)\w*")
HIGH = re.compile(r"\b(moderate|large|marked|massive|severe|extensive|significant|belirgin|ileri|izrazit|μετρι|голям)\w*")

PATTERNS = {
    "ACL": re.compile(r"\bacl\b|anterior cruciate|cruzado anterior|croise anterieur|voorste kruisband|vorder\w* kreuzband|on capraz|prednji krizni|χιαστ|предна кръстна"),
    "MCL": re.compile(r"\bmcl\b|medial collateral|tibial collateral|colateral medial|collateral medial|innenband|medial\w* kollateral|ic yan bag|medijalni kolateral|εσω πλαγι|медиален колатерал"),
    "Medial Meniscus": re.compile(r"medial\w* menisc|menisc\w* medial|menisco interno|menisque interne|binnenmenisc|innenmenisk|medyal menisk|ic menisk|medijalni menisk|εσω μηνισκ|медиал\w* менискус"),
    "Lateral Meniscus": re.compile(r"lateral\w* menisc|menisc\w* lateral|menisco externo|menisque externe|buitenmenisc|aussenmenisk|dis menisk|lateralni menisk|εξω μηνισκ|латерал\w* менискус"),
    "Medial OA": re.compile(r"medial\w* (tibiofemoral|femorotibial|compartment|condyle)|compartimento femorotibial medial|εσω διαμερισμα|медиал\w* (компарт|отдел)"),
    "Lateral OA": re.compile(r"lateral\w* (tibiofemoral|femorotibial|compartment|condyle)|compartimento femorotibial lateral|εξω διαμερισμα|латерал\w* (компарт|отдел)"),
    "PF OA": re.compile(r"patello?femoral|femoropatel|retropatellar|retrorotul|trochlea|chondromalacia patella|μηροεπιγονατιδ|феморопател"),
    "Effusion": re.compile(r"effusion|joint fluid|hydrops|derrame articular|epanchement|gewrichtsvocht|gelenkerguss|efuzyon|eklem\w* sivi|izliv|αρθρικ\w* υγρ|ставен излив"),
    "Synovitis": re.compile(r"synovit|sinovit|synovial (thickening|proliferation|hypertroph)|υμενιτιδα|синовит"),
    "Baker's": re.compile(r"baker|popliteal cyst|quiste poplite|kyste poplite|poplitealzyste|popliteal kist|κυστη baker|бейкърова киста"),
    "Contusion": re.compile(r"bone bruise|bone marrow (edema|oedema|contusion)|osseous contusion|contusion osea|knochenmarkodem|kemik kontuzyonu|kostani edem|οστικο οιδημα|костномозъчен едем"),
    "Fracture": re.compile(r"fractur|fractura|fractuur|\bbreuk\b|fraktur|\bbruch\b|\bkirik|prijelom|καταγμα|фрактур|счупван"),
}


def report_rule_scores(report: object) -> Tuple[np.ndarray, np.ndarray, List[str]]:
    text = norm_report(report)
    clauses = [c.strip() for c in re.split(r"(?<=[.;!?])\s+|\n+", text) if c.strip()]
    y, w, states = [], [], []
    for target in TARGETS:
        mentions = [c for c in clauses if PATTERNS[target].search(c)]
        if target in {"ACL", "MCL", "Medial Meniscus", "Lateral Meniscus"}:
            mentions = [c for c in mentions if PATH.search(c) or NEG.search(c) or NORMAL.search(c)]
        elif target in {"Medial OA", "Lateral OA", "PF OA"}:
            mentions = [c for c in mentions if OA.search(c) or NEG.search(c) or NORMAL.search(c)]
        pos, neg, uncertain = [], [], []
        for c in mentions:
            if UNC.search(c):
                uncertain.append(c)
            elif NEG.search(c) or (NORMAL.search(c) and not PATH.search(c) and not OA.search(c)):
                neg.append(c)
            else:
                pos.append(c)
        if pos:
            score = max(0.94 if HIGH.search(c) else 0.82 if not LOW.search(c) else 0.68 for c in pos)
            conf, state = 0.95, "YES"
        elif uncertain:
            score, conf, state = 0.68, 0.85, "UNCERTAIN"
        elif neg:
            score, conf, state = 0.08, 0.95, "NO"
        else:
            score, conf, state = 0.28, 0.05, "UNK"
        y.append(score); w.append(conf); states.append(state)
    y = np.asarray(y, np.float32)
    w = np.asarray(w, np.float32)
    # Report silence for synovitis is weak evidence; effusion is a useful proxy.
    j_syn, j_eff = TARGETS.index("Synovitis"), TARGETS.index("Effusion")
    if states[j_syn] == "UNK":
        y[j_syn] = max(y[j_syn], 0.28 + 0.45 * (y[j_eff] - 0.28))
        w[j_syn] = max(w[j_syn], 0.20 if states[j_eff] != "UNK" else 0.05)
    return y, w, states


def find_exact_label_table(configured: Optional[str], filename: str,
                           dataset_slug: str) -> Optional[Path]:
    if configured:
        path = Path(configured)
        if not path.is_file():
            raise FileNotFoundError(f"configured label table does not exist: {path}")
        if path.name.lower() != filename.lower():
            raise ValueError(f"expected {filename}, got {path.name}")
        return path
    base = Path("/kaggle/input")
    if not base.is_dir():
        return None
    matches = []
    for root, dirs, files in os.walk(base):
        dirs[:] = [d for d in dirs if d not in ("train_series", "test_series", ".git")]
        for file in files:
            if file.lower() == filename.lower():
                matches.append(Path(root) / file)
    matches = sorted(set(matches), key=lambda path: str(path))
    preferred = [path for path in matches if dataset_slug in str(path).lower()]
    if len(preferred) == 1:
        return preferred[0]
    if len(matches) == 1:
        return matches[0]
    if len(matches) > 1:
        listed = "\n".join(f"  - {path}" for path in matches)
        raise RuntimeError(
            f"multiple copies of {filename}; set the corresponding environment path:\n{listed}"
        )
    return None


def load_label_source(path: Path, default_confidence: float) -> dict:
    raw = pd.read_csv(path, dtype={UID: str})
    required = [UID] + TARGETS
    missing = [column for column in required if column not in raw.columns]
    if missing:
        raise ValueError(f"missing columns in {path}: {missing}")
    if raw[UID].isna().any() or raw[UID].duplicated().any():
        raise ValueError(f"label table must have unique, non-null {UID}: {path}")
    table = raw.set_index(UID)
    score = table[TARGETS].apply(pd.to_numeric, errors="coerce").to_numpy(np.float32)
    valid = np.isfinite(score)
    if (valid & ((score < 0) | (score > 1))).any():
        raise ValueError(f"scores outside [0, 1] in {path}")
    conf_cols = [target + "__conf" for target in TARGETS]
    verdict_cols = [target + "__verdict" for target in TARGETS]
    if all(column in table.columns for column in conf_cols):
        confidence = table[conf_cols].apply(pd.to_numeric, errors="coerce").to_numpy(np.float32)
        confidence = np.where(np.isfinite(confidence), np.clip(confidence, 0, 1), default_confidence)
    else:
        confidence = np.full_like(score, default_confidence)
    if all(column in table.columns for column in verdict_cols):
        verdict = table[verdict_cols].fillna("UNK").astype(str).to_numpy(object)
        verdict = np.char.upper(verdict.astype(str)).astype(object)
    else:
        verdict = np.where(confidence <= 0.10, "UNK", np.where(score >= 0.5, "YES", "NO"))
    return {"path": path, "table": table, "score": score, "confidence": confidence,
            "verdict": verdict}


def align_source(frame: pd.DataFrame, record: dict) -> Tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray]:
    table = record["table"]
    n, labels = len(frame), len(TARGETS)
    score = np.full((n, labels), np.nan, np.float32)
    confidence = np.zeros((n, labels), np.float32)
    verdict = np.full((n, labels), "UNK", dtype=object)
    hit = frame[UID].isin(table.index).to_numpy()
    row_ids = frame.loc[hit, UID].tolist()
    score[hit] = table.loc[row_ids, TARGETS].apply(pd.to_numeric, errors="coerce").to_numpy(np.float32)
    conf_cols = [target + "__conf" for target in TARGETS]
    verdict_cols = [target + "__verdict" for target in TARGETS]
    if all(column in table.columns for column in conf_cols):
        values = table.loc[row_ids, conf_cols].apply(pd.to_numeric, errors="coerce").to_numpy(np.float32)
        confidence[hit] = np.where(np.isfinite(values), np.clip(values, 0, 1), 0.85)
    else:
        confidence[hit] = 0.85
    if all(column in table.columns for column in verdict_cols):
        values = table.loc[row_ids, verdict_cols].fillna("UNK").astype(str).to_numpy(object)
        verdict[hit] = np.char.upper(values.astype(str)).astype(object)
    else:
        verdict[hit] = np.where(confidence[hit] <= 0.10, "UNK",
                                np.where(score[hit] >= 0.5, "YES", "NO"))
    return score, confidence, verdict, hit


def build_pseudo_labels(frame: pd.DataFrame) -> Tuple[np.ndarray, np.ndarray, np.ndarray, pd.DataFrame]:
    rule = [report_rule_scores(value) for value in frame["Report"].fillna("")]
    rule_y = np.stack([item[0] for item in rule])
    rule_c = np.stack([item[1] for item in rule])
    rule_states = np.asarray([item[2] for item in rule], dtype=object)
    y, conf, verdict = rule_y.copy(), rule_c.copy(), rule_states.copy()
    source = np.full(y.shape, "rules", dtype=object)

    pilkwang_path = find_exact_label_table(
        cfg.pilkwang_label_path, "report_labels_v2.csv", "rsna-knee-llm-labels"
    )
    steven_path = find_exact_label_table(
        cfg.steven_label_path, "llm_labels_v4_blend.csv", "rsna-knee-llm-report-labels"
    )
    if cfg.require_llm_labels and not cfg.smoke and (pilkwang_path is None or steven_path is None):
        raise FileNotFoundError(
            "V7B requires both datasets: pilkwang/rsna-knee-llm-labels/report_labels_v2.csv "
            "and stevenleehans/rsna-knee-llm-report-labels/llm_labels_v4_blend.csv"
        )
    records = {}
    if pilkwang_path is not None:
        records["pilkwang"] = load_label_source(pilkwang_path, 0.85)
    if steven_path is not None:
        records["steven_v4"] = load_label_source(steven_path, 0.85)

    aligned = {}
    for name, record in records.items():
        aligned[name] = align_source(frame, record)
        print(f"{name} labels: {record['path']} coverage={int(aligned[name][3].sum())}/{len(frame)}")

    if "pilkwang" in aligned and "steven_v4" in aligned:
        py, pc, pv, ph = aligned["pilkwang"]
        sy, sc, sv, sh = aligned["steven_v4"]
        p_valid, s_valid = np.isfinite(py), np.isfinite(sy)
        both = p_valid & s_valid
        only_p, only_s = p_valid & ~s_valid, s_valid & ~p_valid
        alpha = float(cfg.steven_blend_alpha)
        if not 0 <= alpha <= 1:
            raise ValueError(f"steven_blend_alpha must be in [0, 1], got {alpha}")
        y[both] = (1.0 - alpha) * py[both] + alpha * sy[both]
        conf[both] = (1.0 - alpha) * pc[both] + alpha * sc[both]
        verdict[both] = np.where(conf[both] <= 0.10, "UNK", np.where(y[both] >= 0.5, "YES", "NO"))
        source[both] = "pilkwang+steven_v4"
        y[only_p], conf[only_p], verdict[only_p], source[only_p] = (
            py[only_p], pc[only_p], pv[only_p], "pilkwang"
        )
        y[only_s], conf[only_s], verdict[only_s], source[only_s] = (
            sy[only_s], sc[only_s], sv[only_s], "steven_v4"
        )
        print("blended cells:", int(both.sum()), "pilkwang-only:", int(only_p.sum()),
              "steven-only:", int(only_s.sum()))
    elif "pilkwang" in aligned or "steven_v4" in aligned:
        name = next(iter(aligned))
        values, weights, states, _ = aligned[name]
        valid = np.isfinite(values)
        y[valid], conf[valid], verdict[valid], source[valid] = (
            values[valid], weights[valid], states[valid], name
        )
        print(f"SMOKE/fallback: using only {name}")
    else:
        print("SMOKE: no LLM tables attached; using multilingual rules")

    gold = frame[TARGETS].notna().all(axis=1).to_numpy()
    if cfg.pseudo_temperature < 1.0:
        probability = np.clip(y, 1e-4, 1 - 1e-4)
        sharpened = 1.0 / (1.0 + np.exp(
            -np.log(probability / (1 - probability)) / cfg.pseudo_temperature
        ))
        mix = np.square(np.clip(conf, 0, 1))
        eligible = (~gold[:, None]) & (np.char.upper(verdict.astype(str)) != "UNK")
        y[eligible] = ((1 - mix) * probability + mix * sharpened)[eligible]

    audit, audit_rows = [], []
    for index, target in enumerate(TARGETS):
        truth = frame.loc[gold, target].to_numpy(np.int64)
        prediction = y[gold, index]
        auc = roc_auc_score(truth, prediction) if len(np.unique(truth)) > 1 else np.nan
        audit.append(auc)
        gold_unk = np.char.upper(verdict[gold, index].astype(str)) == "UNK"
        audit_rows.append({
            "target": target, "gold_auc": auc,
            "pseudo_positive_rate": float((y[:, index] >= 0.5).mean()),
            "unk_fraction": float((np.char.upper(verdict[:, index].astype(str)) == "UNK").mean()),
            "gold_positive_when_unk": float(truth[gold_unk].mean()) if gold_unk.any() else np.nan,
        })
    print("V7B blended pseudo-label gold macro AUC:", round(float(np.nanmean(audit)), 4))
    print(pd.DataFrame(audit_rows).round(4).to_string(index=False))

    weight = (0.05 + 0.95 * np.square(np.clip(conf, 0, 1))).astype(np.float32)
    unknown = np.char.upper(verdict.astype(str)) == "UNK"
    weight[unknown] = np.minimum(weight[unknown], cfg.unknown_weight)
    diag = frame[[UID, "Report"]].copy()
    diag["is_gold"] = gold
    diag["report_hash"] = frame["Report"].fillna("").map(
        lambda value: hashlib.sha1(norm_report(value).encode()).hexdigest()
    )
    for index, target in enumerate(TARGETS):
        diag[f"y__{target}"] = y[:, index]
        diag[f"w__{target}"] = weight[:, index]
        diag[f"conf__{target}"] = conf[:, index]
        diag[f"verdict__{target}"] = verdict[:, index]
        diag[f"source__{target}"] = source[:, index]
        diag[f"rule_state__{target}"] = rule_states[:, index]
    return y.astype(np.float32), weight.astype(np.float32), conf.astype(np.float32), diag


Y, W, C, pseudo_df = build_pseudo_labels(train_df)
pseudo_path = OUT / "pseudo_labels.csv"
pseudo_df.to_csv(pseudo_path, index=False)
gold_mask = pseudo_df["is_gold"].to_numpy(bool)
print("gold studies:", int(gold_mask.sum()), "/", len(train_df))
print("pseudo label file:", pseudo_path)

## 3. Public 44-slice Raptor corpus and study-level 2.5D sampling

In [ ]:
class TwoPartVolumes:
    def __init__(self, first, second):
        self.first, self.second = first, second
        self.n_first = first.shape[0]
        self.shape = (first.shape[0] + second.shape[0],) + tuple(first.shape[1:])

    def __len__(self): return self.shape[0]

    def __getitem__(self, index):
        return self.first[index] if index < self.n_first else self.second[index - self.n_first]


def find_input_file(filename: str, preferred_slug: str = "") -> Optional[Path]:
    base = Path("/kaggle/input")
    if not base.is_dir(): return None
    matches = []
    for root, dirs, files in os.walk(base):
        dirs[:] = [d for d in dirs if d not in ("train_series", "test_series")]
        if filename in files: matches.append(Path(root) / filename)
    preferred = [p for p in matches if preferred_slug and preferred_slug in str(p).lower()]
    if len(preferred) == 1: return preferred[0]
    if len(matches) == 1: return matches[0]
    if len(matches) > 1:
        raise RuntimeError(f"multiple {filename} files found; attach only the Raptor corpus copies: {matches}")
    return None


def smoke_corpus():
    ids = train_df[UID].astype(str).to_numpy()
    n, depth, side = len(ids), 44, cfg.corpus_size
    yy, xx = np.mgrid[:side, :side]
    vols = np.empty((n, depth, side, side), np.uint8)
    for i in range(n):
        for z in range(depth):
            value = 80 + 45 * np.sin((xx + i) / 7) + 35 * np.cos((yy + z) / 9)
            vols[i, z] = np.clip(value + ((i + z) % 5) * 8, 0, 255).astype(np.uint8)
    return vols, np.ones((n, depth), np.uint8), ids


def open_corpus():
    if cfg.smoke:
        return smoke_corpus()
    av = find_input_file("all_vols.npy", "knee-raptor-corpus")
    bv = find_input_file("extra_vols.npy", "knee-raptor-corpus-ext")
    am = find_input_file("all_masks.npy", "knee-raptor-corpus")
    bm = find_input_file("extra_masks.npy", "knee-raptor-corpus-ext")
    ai = find_input_file("all_ids.npy", "knee-raptor-corpus")
    bi = find_input_file("extra_ids.npy", "knee-raptor-corpus-ext")
    if not all((av, bv, am, bm, ai, bi)):
        raise FileNotFoundError(
            "Attach both public datasets: dreaddevelopment/knee-raptor-corpus and "
            "dreaddevelopment/knee-raptor-corpus-ext"
        )
    vols = TwoPartVolumes(np.load(av, mmap_mode="r"), np.load(bv, mmap_mode="r"))
    masks = np.concatenate([np.load(am), np.load(bm)], axis=0)
    ids = np.concatenate([np.load(ai, allow_pickle=True), np.load(bi, allow_pickle=True)]).astype(str)
    return vols, masks, ids


VOLS, MASKS, CORPUS_IDS = open_corpus()
assert len(CORPUS_IDS) == len(VOLS) == len(MASKS)
assert len(set(CORPUS_IDS)) == len(CORPUS_IDS)
assert VOLS.shape[1] == 44 and MASKS.shape[1] == 44
if not cfg.smoke:
    assert tuple(VOLS.shape[2:]) == (336, 336), VOLS.shape
ID_TO_ROW = {uid: i for i, uid in enumerate(CORPUS_IDS)}
print("corpus:", VOLS.shape, "mask:", MASKS.shape, "ids:", len(CORPUS_IDS))


class StudyWindows(Dataset):
    """Public Raptor sampling: random K 2.5D windows in train, even K in eval."""
    def __init__(self, ids: List[str], labels: Dict[str, np.ndarray], k: int, train: bool):
        self.ids, self.labels, self.k, self.train = list(ids), labels, int(k), bool(train)

    def __len__(self): return len(self.ids)

    def centers(self, valid: np.ndarray) -> List[int]:
        lo, hi = int(valid.min()), int(valid.max())
        candidates = [c for c in range(lo + 1, hi) if c - 1 >= lo and c + 1 <= hi]
        if not candidates: candidates = [max(1, min((lo + hi) // 2, VOLS.shape[1] - 2))]
        if self.train:
            pool = candidates * (self.k // len(candidates) + 1)
            random.shuffle(pool)
            return pool[:self.k]
        pos = np.linspace(0, len(candidates) - 1, self.k).round().astype(int)
        return [candidates[j] for j in pos]

    def __getitem__(self, index: int):
        uid = self.ids[index]; row = ID_TO_ROW[uid]
        valid = np.flatnonzero(MASKS[row] > 0)
        if len(valid) < 3: valid = np.arange(min(3, VOLS.shape[1]))
        vol = VOLS[row]
        tiles = []
        for center in self.centers(valid):
            center = max(1, min(int(center), VOLS.shape[1] - 2))
            tri = np.stack([vol[center - 1], vol[center], vol[center + 1]], 0).astype(np.float32) / 255.0
            x = torch.from_numpy(np.ascontiguousarray(tri))
            if x.shape[-2:] != (cfg.resolution, cfg.resolution):
                x = F.interpolate(x[None], (cfg.resolution, cfg.resolution),
                                  mode="bilinear", align_corners=False)[0]
            tiles.append(x)
        x = torch.stack(tiles)
        if self.train and cfg.intensity_jitter > 0:
            gain = 1.0 + (random.random() - 0.5) * cfg.intensity_jitter
            x = (x * gain).clamp(0, 1)
        return x, torch.from_numpy(self.labels[uid])


def collate(batch):
    return torch.stack([x for x, _ in batch]), torch.stack([y for _, y in batch])

## 4. Single CoAtNet + pathology-specific attention

In [ ]:
class RaptorClassifier(nn.Module):
    """One CoAtNet encoder plus pathology-specific attention pooling."""
    def __init__(self, backbone: nn.Module, feature_dim: int, n_targets: int = 12, dropout: float = 0.2):
        super().__init__()
        self.backbone = backbone
        self.norm = nn.LayerNorm(feature_dim)
        self.att = nn.Sequential(
            nn.Linear(feature_dim, 256), nn.Tanh(), nn.Dropout(dropout), nn.Linear(256, n_targets)
        )
        self.cls_weight = nn.Parameter(torch.zeros(n_targets, feature_dim))
        self.cls_bias = nn.Parameter(torch.zeros(n_targets))
        nn.init.trunc_normal_(self.cls_weight, std=0.02)

    def encode(self, x):
        batch, windows = x.shape[:2]
        feature = self.backbone(x.flatten(0, 1))
        return feature.reshape(batch, windows, -1)

    def head(self, feature):
        hidden = self.norm(feature)
        attention = torch.softmax(self.att(hidden), dim=1)
        pooled = torch.einsum("bkn,bkf->bnf", attention, hidden)
        return (pooled * self.cls_weight).sum(-1) + self.cls_bias

    def forward(self, x):
        return self.head(self.encode(x))



class TinyBackbone(nn.Module):
    num_features = 32
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(3, 16, 3, 2, 1), nn.GELU(),
            nn.Conv2d(16, 32, 3, 2, 1), nn.GELU(), nn.AdaptiveAvgPool2d(1), nn.Flatten(),
        )
    def forward(self, x): return self.net(x)


def build_backbone(pretrained: bool):
    if cfg.smoke:
        return TinyBackbone()
    model = timm.create_model(cfg.arch, pretrained=pretrained, num_classes=0,
                              in_chans=3, global_pool="avg")
    if cfg.grad_checkpointing:
        try:
            model.set_grad_checkpointing(True)
            print("gradient checkpointing: ON")
        except Exception as exc:
            print("WARNING: gradient checkpointing unsupported:", exc)
    return model


def unwrap(model): return model.module if isinstance(model, nn.DataParallel) else model


def cpu_state(model):
    return {key: value.detach().cpu().clone() for key, value in unwrap(model).state_dict().items()}


def macro_auc(y: np.ndarray, p: np.ndarray):
    per_target = {}
    for j, target in enumerate(TARGETS):
        if len(np.unique(y[:, j].astype(int))) > 1:
            per_target[target] = float(roc_auc_score(y[:, j], p[:, j]))
    return float(np.mean(list(per_target.values()))), per_target


gold_ids = [u for u in train_df.loc[gold_mask, UID].astype(str) if u in ID_TO_ROW]
pseudo_ids = [u for u in CORPUS_IDS if u in set(train_df.loc[~gold_mask, UID].astype(str))]
row_for_uid = {u: i for i, u in enumerate(train_df[UID].astype(str))}
labels = {u: Y[row_for_uid[u]].astype(np.float32) for u in pseudo_ids}
for u in gold_ids:
    labels[u] = train_df.loc[row_for_uid[u], TARGETS].to_numpy(np.float32)
if not cfg.smoke:
    assert len(gold_ids) == 58, len(gold_ids)
    assert len(pseudo_ids) >= 4300, len(pseudo_ids)
print("training pseudo studies:", len(pseudo_ids), "gold validation studies:", len(gold_ids))

train_ds = StudyWindows(pseudo_ids, labels, cfg.train_windows, train=True)
valid_ds = StudyWindows(gold_ids, labels, cfg.eval_windows, train=False)
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=cfg.workers,
                          drop_last=True, collate_fn=collate, pin_memory=torch.cuda.is_available(),
                          persistent_workers=cfg.workers > 0)
valid_loader = DataLoader(valid_ds, batch_size=cfg.valid_batch_size, shuffle=False,
                          num_workers=cfg.workers, collate_fn=collate,
                          pin_memory=torch.cuda.is_available(), persistent_workers=cfg.workers > 0)
print("train studies/batch:", BATCH_SIZE, "train windows/study:", cfg.train_windows,
      "effective backbone images/global step:", BATCH_SIZE * cfg.train_windows)
print("validation studies/batch:", cfg.valid_batch_size, "eval windows/study:", cfg.eval_windows)

backbone = build_backbone(pretrained=not cfg.smoke)
feature_dim = int(backbone.num_features)
core = RaptorClassifier(backbone, feature_dim, len(TARGETS), cfg.dropout).to(DEVICE)
if not all(parameter.requires_grad for parameter in core.backbone.parameters()):
    raise RuntimeError("V7B contract violation: the complete CoAtNet backbone must be trainable")
probe_name, probe_parameter = next(core.backbone.named_parameters())
probe_before = probe_parameter.detach().cpu().float().clone()
print("fully trainable backbone parameters:",
      sum(parameter.numel() for parameter in core.backbone.parameters()) / 1e6, "M")
if N_GPU >= 2:
    model = nn.DataParallel(core, device_ids=list(range(N_GPU)), output_device=0)
    print("DataParallel active on:", list(range(N_GPU)))
else:
    model = core
    print("single-device training")

head_params = [p for name, p in core.named_parameters() if not name.startswith("backbone.")]
backbone_params = [p for name, p in core.named_parameters() if name.startswith("backbone.")]
optimizer = torch.optim.AdamW([
    {"params": backbone_params, "lr": cfg.backbone_lr},
    {"params": head_params, "lr": cfg.head_lr},
], weight_decay=cfg.weight_decay)
total_steps = max(1, len(train_loader) * cfg.epochs)
scheduler = torch.optim.lr_scheduler.OneCycleLR(
    optimizer, max_lr=[cfg.backbone_lr, cfg.head_lr], total_steps=total_steps, pct_start=0.15
)
prevalence = np.clip(np.stack([labels[u] for u in pseudo_ids]).mean(0), 0.03, 0.70)
pos_weight = torch.tensor(np.clip((1 - prevalence) / prevalence, 1, 10),
                          dtype=torch.float32, device=DEVICE)
loss_fn = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
scaler = torch.amp.GradScaler("cuda", enabled=DEVICE.type == "cuda")


@torch.no_grad()
def evaluate():
    model.eval(); predictions, truth = [], []
    for x, y in valid_loader:
        x = x.to(DEVICE, non_blocking=True)
        with torch.autocast("cuda", dtype=torch.float16, enabled=DEVICE.type == "cuda"):
            pred = torch.sigmoid(model(x)).float()
        predictions.append(pred.cpu().numpy()); truth.append(y.numpy())
        del x, y, pred
    p, y = np.concatenate(predictions), np.concatenate(truth)
    score, per_target = macro_auc(y, p)
    return score, per_target, p, y


history, top = [], []
best_score, best_state, best_pred = -1.0, None, None
started = time.time()
for epoch in range(cfg.epochs):
    model.train(); total_loss = 0.0; skipped_amp_steps = 0
    if torch.cuda.is_available():
        for gpu_index in range(N_GPU):
            torch.cuda.reset_peak_memory_stats(gpu_index)
    for step, (x, y) in enumerate(train_loader, start=1):
        x = x.to(DEVICE, non_blocking=True); y = y.to(DEVICE, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        with torch.autocast("cuda", dtype=torch.float16, enabled=DEVICE.type == "cuda"):
            logits = model(x)
            loss = loss_fn(logits.float(), y)
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        nn.utils.clip_grad_norm_(model.parameters(), 3.0)

        # GradScaler can skip optimizer.step() when FP16 overflow is detected.
        # Only advance OneCycleLR when the optimizer really stepped.
        scale_before = scaler.get_scale()
        scaler.step(optimizer)
        scaler.update()
        scale_after = scaler.get_scale()
        if scale_after >= scale_before:
            scheduler.step()
        else:
            skipped_amp_steps += 1

        total_loss += float(loss.detach())
        # Do not keep the last training graph alive into validation.
        del x, y, logits, loss

    train_minutes = (time.time() - started) / 60
    peak_vram = ([round(torch.cuda.max_memory_allocated(i) / (1024**3), 2) for i in range(N_GPU)]
                 if torch.cuda.is_available() else [])
    print(f"epoch={epoch+1:02d} train_done loss={total_loss/max(len(train_loader),1):.4f} "
          f"amp_skips={skipped_amp_steps} peak_vram_gb={peak_vram} "
          f"elapsed={train_minutes:.1f}m -> validating", flush=True)
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    score, per_target, pred, truth = evaluate()

    # Release any cached validation allocations before checkpoint materialization.
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    record = {"epoch": epoch + 1, "loss": total_loss / max(len(train_loader), 1),
              "gold_auc": score, "amp_skipped_steps": skipped_amp_steps,
              "elapsed_minutes": (time.time() - started) / 60}
    history.append(record)
    state = cpu_state(model)
    candidate = {"score": score, "epoch": epoch + 1, "state": state,
                 "per_target": per_target, "prediction": pred}
    top.append(candidate); top.sort(key=lambda item: -item["score"]); top = top[:cfg.top_k]
    if score > best_score:
        best_score, best_state, best_pred = score, state, pred
        torch.save({"state": best_state, "epoch": epoch + 1, "gold_auc": score},
                   OUT / "v7b_best_resume.pt")
    print(f"epoch={epoch+1:02d} loss={record['loss']:.4f} gold_auc={score:.4f} "
          f"best={best_score:.4f} elapsed={record['elapsed_minutes']:.1f}m", flush=True)
    gc.collect()

if not top:
    raise RuntimeError("training produced no checkpoint")
average_state = {}
for key in top[0]["state"]:
    value = top[0]["state"][key]
    if value.is_floating_point():
        acc = value.float().clone()
        for item in top[1:]: acc.add_(item["state"][key].float())
        average_state[key] = acc.div_(len(top)).to(value.dtype)
    else:
        average_state[key] = value.clone()
unwrap(model).load_state_dict(average_state)
swa_score, swa_per_target, swa_pred, truth = evaluate()
if swa_score >= best_score:
    selected_state, selected_score, selected_pred, selected_kind = (
        average_state, swa_score, swa_pred, "top-k-weight-average"
    )
    selected_per_target = swa_per_target
else:
    selected_state, selected_score, selected_pred, selected_kind = (
        best_state, best_score, best_pred, "best-epoch"
    )
    selected_per_target = top[0]["per_target"]
probe_after = selected_state["backbone." + probe_name].float()
backbone_probe_delta = float((probe_after - probe_before).abs().mean())
if not np.isfinite(backbone_probe_delta) or backbone_probe_delta <= 0:
    raise RuntimeError(f"full end-to-end backbone produced no update: {backbone_probe_delta}")
print("top epochs:", [(item["epoch"], round(item["score"], 5)) for item in top])
print("SWA gold AUC:", swa_score, "selected:", selected_kind, selected_score)
print("backbone probe update:", probe_name, backbone_probe_delta)

contract = {
    "slot_specs": SLOT_SPECS, "tokens_per_slot": cfg.tokens_per_slot,
    "corpus_size": cfg.corpus_size, "crop_mm": cfg.crop_mm, "coverage": cfg.coverage,
    "input_mode": "public-contiguous-stack-adjacent-2p5d",
    "resolution": cfg.resolution, "eval_windows": cfg.eval_windows,
}
signature = hashlib.sha256(json.dumps(contract, sort_keys=True).encode()).hexdigest()
bundle = {
    "version": "rsna-knee-v7b-coatnet-raptor-single-method",
    "model_schema": "raptor-coatnet-target-attention-v1",
    "targets": TARGETS, "arch": "tiny" if cfg.smoke else cfg.arch,
    "feature_dim": feature_dim, "dropout": cfg.dropout,
    **contract, "preprocess_signature": signature,
    "model_state": selected_state,
    "selection": {"kind": selected_kind, "gold_auc": selected_score,
                  "per_target": selected_per_target,
                  "top_epochs": [(item["epoch"], item["score"]) for item in top]},
    "training": {"epochs": cfg.epochs, "backbone_lr": cfg.backbone_lr,
                 "head_lr": cfg.head_lr, "weight_decay": cfg.weight_decay,
                 "train_windows": cfg.train_windows, "global_batch": BATCH_SIZE,
                 "full_backbone_tune": True, "backbone_probe": probe_name,
                 "backbone_probe_delta": backbone_probe_delta, "history": history,
                 "n_pseudo": len(pseudo_ids), "n_gold_optimizer": 0},
    "pseudo_supervision": {"pilkwang": "report_labels_v2.csv",
                           "steven": "llm_labels_v4_blend.csv", "blend": 0.5},
}
bundle_path = OUT / "rsna_knee_coatnet_v7b_bundle.pt"
torch.save(bundle, bundle_path)
gold_out = pd.DataFrame({UID: gold_ids})
gold_out[TARGETS] = selected_pred
for j, target in enumerate(TARGETS): gold_out[f"y__{target}"] = truth[:, j]
gold_out.to_csv(OUT / "gold_predictions_v7b.csv", index=False)
with open(OUT / "metrics_v7b.json", "w", encoding="utf-8") as f:
    json.dump({"selected_gold_auc": selected_score, "selection": selected_kind,
               "per_target": selected_per_target, "history": history}, f, indent=2)
(OUT / "v7b_best_resume.pt").unlink(missing_ok=True)
print("saved:", bundle_path, f"{bundle_path.stat().st_size/1e6:.1f} MB")
print("preprocess signature:", signature)

## Outputs

- `rsna_knee_coatnet_v7b_bundle.pt`: one selected CoAtNet state.
- `gold_predictions_v7b.csv`: predictions on the 58 validation studies.
- `metrics_v7b.json`: epoch history, AMP skipped-step count, and per-target AUC.

### T4×2 safety changes in this revision

- `batch_per_gpu=1` → global batch 2.
- `train_windows=6`, `eval_windows=12`.
- Validation batch is fixed to 1 study.
- Last training graph is explicitly deleted before validation.
- `gc.collect()` + `torch.cuda.empty_cache()` run at the train→validation boundary.
- OneCycleLR steps only when GradScaler actually performs the optimizer update.
- A `train_done -> validating` log line makes future crashes phase-localizable.

This intentionally does not use OrthoFoundation, V6 supervision modules, multi-architecture fusion, or prediction-level checkpoint ensembles.
